# 🎬 Anime Factory v2 — Wan2GP Batch Worker
**GPU required:** Runtime → Change runtime type → T4 GPU

Run cells **1 → 7** in order. This notebook uses [Wan2GP](https://github.com/deepbeepmeep/Wan2GP) instead of ComfyUI — it's faster, uses less VRAM, and supports Wan 2.1 + LTX-2 from one codebase.

## Cell 1 — Mount Drive & Install System Dependencies

In [ ]:
# CELL 1: Mount Drive & install system tools
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess, sys, os
print('Installing system tools...')
subprocess.run(['apt-get', '-qq', 'install', '-y', 'aria2', 'ffmpeg'],
               check=True, capture_output=True)
print('Installing Python packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'gradio_client', 'requests'],
               check=True, capture_output=True)
print('✅ Cell 1 complete!')

## Cell 2 — Clone & Install Wan2GP

In [ ]:
# CELL 2: Clone Wan2GP (skip if already present)
import os

WAN2GP_DIR = '/content/Wan2GP'

if not os.path.exists(WAN2GP_DIR):
    print('Cloning Wan2GP...')
    !git clone --depth 1 https://github.com/deepbeepmeep/Wan2GP.git {WAN2GP_DIR}
    %cd {WAN2GP_DIR}
    print('Installing Wan2GP dependencies...')
    !pip install -q -r requirements.txt
    print('✅ Wan2GP installed!')
else:
    print('✅ Wan2GP already present.')
    %cd {WAN2GP_DIR}

# Create output directory
os.makedirs(f'{WAN2GP_DIR}/outputs', exist_ok=True)
print(f'Output dir: {WAN2GP_DIR}/outputs')

## Cell 2b — 🩺 Diagnostics (run if server fails to start)

In [ ]:
# CELL 2b: Diagnostics — run this FIRST if Cell 3 (server) fails
# It checks GPU, torch/CUDA, import health, and dependency conflicts.
import subprocess, sys, os

WAN2GP_DIR = '/content/Wan2GP'

print('=' * 60)
print('  GPU / VRAM')
print('=' * 60)
gpu = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu.returncode == 0 and gpu.stdout.strip():
    print(gpu.stdout.strip())
else:
    print('❌ nvidia-smi failed or no GPU attached')
    if gpu.stderr.strip():
        print(gpu.stderr.strip()[:1000])

print('\n' + '=' * 60)
print('  Python / CUDA versions')
print('=' * 60)
subprocess.run([sys.executable, '-c',
    'import torch; print("torch:", torch.__version__); '
    'print("CUDA available:", torch.cuda.is_available()); '
    'print("CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")'
], cwd=WAN2GP_DIR)

print('\n' + '=' * 60)
print('  wgp.py import smoke test')
print('=' * 60)
try:
    result = subprocess.run(
        [sys.executable, '-c', 'import wgp; print("import OK")'],
        cwd=WAN2GP_DIR, capture_output=True, text=True, timeout=90
    )
    if 'import OK' in result.stdout:
        print('✅ import wgp succeeded')
    else:
        print('❌ import wgp failed')
        print('STDOUT:', result.stdout[-3000:])
        print('STDERR:', result.stderr[-3000:])
except subprocess.TimeoutExpired as e:
    print('⚠️ import test timed out (>90s). This can happen on slow first-run init.')
    if e.stdout:
        print('PARTIAL STDOUT:', str(e.stdout)[-2000:])
    if e.stderr:
        print('PARTIAL STDERR:', str(e.stderr)[-2000:])

print('\n' + '=' * 60)
print('  Optional wgp.py --help probe (non-fatal)')
print('=' * 60)
try:
    help_result = subprocess.run(
        [sys.executable, 'wgp.py', '--help'],
        cwd=WAN2GP_DIR, capture_output=True, text=True, timeout=45
    )
    print(f'Exit code: {help_result.returncode}')
    if help_result.stdout.strip():
        print(help_result.stdout[:1200])
    if help_result.stderr.strip():
        print('STDERR:', help_result.stderr[:1200])
except subprocess.TimeoutExpired:
    print('⚠️ wgp.py --help timed out (non-fatal). Continuing diagnostics...')

print('\n' + '=' * 60)
print('  Key package versions')
print('=' * 60)
pkgs = ['gradio', 'diffusers', 'transformers', 'accelerate',
        'imageio', 'einops', 'safetensors', 'ftfy', 'pandas', 'numba', 'pydantic']
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'show', pkg],
                       capture_output=True, text=True)
    version = next((l.split(':')[1].strip() for l in r.stdout.splitlines()
                    if l.startswith('Version')), 'NOT INSTALLED')
    print(f'  {pkg:<20} {version}')

print('\n' + '=' * 60)
print('  Missing requirements check')
print('=' * 60)
req_file = os.path.join(WAN2GP_DIR, 'requirements.txt')
if os.path.exists(req_file):
    r = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
    if r.returncode == 0:
        print('✅ No dependency conflicts')
    else:
        print('⚠️ Dependency issues:')
        print(r.stdout[:4000])
        if r.stderr.strip():
            print(r.stderr[:2000])
else:
    print('requirements.txt not found')

print('\nDone. If Cell 3 fails, share the last 100 lines from /content/wan2gp_server.log')

## Cell 2c — 🔧 Colab dependency repair (run once after Cell 2)

In [ ]:
# CELL 2c: Repair dependency conflicts that commonly break Wan2GP on Colab
import subprocess, sys, os

WAN2GP_DIR = '/content/Wan2GP'
assert os.path.exists(WAN2GP_DIR), 'Run Cell 2 first.'

print('Upgrading pip tooling...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'pip', 'setuptools', 'wheel'], check=True)

print('Reinstalling Wan2GP requirements...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                '-r', f'{WAN2GP_DIR}/requirements.txt'], check=True)

print('Applying Colab compatibility pins...')
# These resolve the exact resolver warnings seen during install and avoid runtime breaks.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                'pandas==2.2.2', 'numba>=0.60,<0.62', 'pydantic>=2.12,<3'], check=True)

print('Running pip check...')
check = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True)
if check.returncode == 0:
    print('✅ Dependency graph is clean')
else:
    print('⚠️ Remaining dependency warnings (usually non-blocking):')
    print(check.stdout[:4000])

print('\n✅ Cell 2c complete. If this is first run, Runtime -> Restart runtime, then run cells 1,2,2c,3.')

## Cell 3 — Start Wan2GP Server (Background)
Wan2GP auto-downloads the correct model for your GPU on first run.
The T4 will use the Wan 2.1 1.3B fp16 model (~3 GB download).

In [ ]:
# CELL 3: Start Wan2GP Gradio server in background
import subprocess, time, urllib.request, threading, os, sys

WAN2GP_DIR = '/content/Wan2GP'
WAN2GP_URL = 'http://127.0.0.1:7860'
LOG_FILE   = '/content/wan2gp_server.log'

# ── Step 0: make sure we're in the right dir ──────────────────────────
os.chdir(WAN2GP_DIR)

# ── Step 1: quick import smoke-test so we see errors immediately ───────
print('🔍 Running import smoke-test...')
result = subprocess.run(
    [sys.executable, '-c', 'import wgp; print("import OK")'],
    cwd=WAN2GP_DIR, capture_output=True, text=True, timeout=60
)
if 'import OK' in result.stdout:
    print('  ✅ wgp.py imports cleanly')
else:
    print('  ❌ Import FAILED — full error below:')
    print('--- STDOUT ---')
    print(result.stdout[-3000:])
    print('--- STDERR ---')
    print(result.stderr[-3000:])
    print('\n⛔ Fix the import error above before continuing.')
    raise SystemExit('wgp.py import failed — see error above')

# ── Step 2: start the server, tee output to log file ──────────────────
# Wan2GP CLI uses --listen (flag), --server-port, and model flags like --t2v-1-3B.
# Do NOT use --port, --lowvram, or --listen 0.0.0.0 (those crash with exit code 2).
WAN2GP_SERVER_CMD = [
    sys.executable, 'wgp.py',
    '--listen',
    '--server-port', '7860',
    '--t2v-1-3B',
    '--output-dir', f'{WAN2GP_DIR}/outputs',
]

log_fh = open(LOG_FILE, 'w')
proc = subprocess.Popen(
    WAN2GP_SERVER_CMD,
    cwd=WAN2GP_DIR,
    stdout=log_fh, stderr=log_fh
)

# Stream log to notebook output in a background thread
def _tail():
    with open(LOG_FILE, 'r') as f:
        while True:
            line = f.readline()
            if line:
                print(line, end='', flush=True)
            else:
                if proc.poll() is not None:
                    break
                time.sleep(0.3)

tail_thread = threading.Thread(target=_tail, daemon=True)
tail_thread.start()

print(f'\nStarting Wan2GP server (logs → {LOG_FILE})...')
print('First run may take 5-10 minutes to download the model.\n')

SERVER_READY = False
for i in range(120):   # up to 10 minutes
    time.sleep(5)
    # Check if process crashed
    if proc.poll() is not None:
        log_fh.flush()
        print(f'\n❌ Wan2GP process exited with code {proc.returncode}!')
        print(f'Full log saved to {LOG_FILE}')
        print('\n--- LAST 100 LINES OF LOG ---')
        with open(LOG_FILE) as f:
            lines = f.readlines()
        print(''.join(lines[-100:]))
        raise SystemExit('Wan2GP crashed — see log above')
    try:
        urllib.request.urlopen(WAN2GP_URL, timeout=3)
        SERVER_READY = True
        print(f'\n✅ Wan2GP is ready! ({(i+1)*5}s)')
        break
    except Exception:
        if i % 6 == 0:
            print(f'  waiting... ({(i+1)*5}s)')

if not SERVER_READY:
    log_fh.flush()
    print(f'\n⚠️  Server still not responding after 10 minutes.')
    print(f'Check {LOG_FILE} for details. Last 50 lines:')
    with open(LOG_FILE) as f:
        lines = f.readlines()
    print(''.join(lines[-50:]))
    print('\nWorker will attempt headless mode instead.')

## Cell 4 — Write drive_state.py (Distributed Lock Manager)

In [ ]:
# CELL 4: Write the Drive state manager
# This is identical to the local version — manages scene claiming,
# heartbeats, and progress tracking via Google Drive filesystem.

import os, shutil

# Copy from Drive if uploaded there, otherwise write inline
DRIVE_ROOT = '/content/drive/MyDrive/AnimeFactory'
local_state = os.path.join(DRIVE_ROOT, 'scripts', 'drive_state.py')

if os.path.exists(local_state):
    shutil.copy2(local_state, '/content/drive_state.py')
    print('✅ drive_state.py copied from Drive')
else:
    # Write the full drive_state.py inline (same as local version)
    state_code = open(os.path.join(
        os.path.dirname(os.path.abspath('__file__')),
        'drive_state.py'
    )).read() if os.path.exists('drive_state.py') else None
    
    if state_code:
        with open('/content/drive_state.py', 'w') as f:
            f.write(state_code)
        print('✅ drive_state.py written from local')
    else:
        print('❌ drive_state.py not found!')
        print('Upload drive_state.py to /content/ or to Drive/AnimeFactory/scripts/')

## Cell 5 — Write wan2gp_worker.py (Batch Video Worker)

In [ ]:
# CELL 5: Write the worker script
# Copy wan2gp_worker.py from Drive or write it inline

import os, shutil

DRIVE_ROOT = '/content/drive/MyDrive/AnimeFactory'
drive_worker = os.path.join(DRIVE_ROOT, 'scripts', 'wan2gp_worker.py')

if os.path.exists(drive_worker):
    shutil.copy2(drive_worker, '/content/wan2gp_worker.py')
    print('✅ wan2gp_worker.py copied from Drive')
elif os.path.exists('/content/wan2gp_worker.py'):
    print('✅ wan2gp_worker.py already present')
else:
    print('❌ wan2gp_worker.py not found!')
    print('Upload the latest file to Drive/AnimeFactory/scripts/wan2gp_worker.py')
    print('Source: slop_generation/video_gen_collab/wan2gp_worker.py')
    print('Important: re-upload after updates (old worker used invalid wgp.py flags).')

## Cell 6 — Verify Drive Structure & Initialize Progress

In [ ]:
# CELL 6: Check Drive structure, create progress.json if needed
import json, os, time

DRIVE_ROOT = '/content/drive/MyDrive/AnimeFactory'

# Create all required folders
for folder in ['state/locks', 'inputs/tts', 'outputs/scenes', 'logs', 'scripts']:
    os.makedirs(os.path.join(DRIVE_ROOT, folder), exist_ok=True)
print('✅ Folder structure ready')

# Check master_script.json
script_path = os.path.join(DRIVE_ROOT, 'state/master_script.json')
if not os.path.exists(script_path):
    print('❌ master_script.json not found!')
    print(f'   Upload to: {script_path}')
    print('   Generate it with: python batch_ai_writer.py --input story.txt')
else:
    with open(script_path) as f:
        scenes = json.load(f)
    print(f'✅ Script loaded: {len(scenes)} scenes')

    # Check TTS files
    tts_dir = os.path.join(DRIVE_ROOT, 'inputs/tts')
    tts_files = [f for f in os.listdir(tts_dir) if f.endswith('.mp3')]
    print(f'✅ TTS audio files: {len(tts_files)}/{len(scenes)}')
    if len(tts_files) < len(scenes):
        print('⚠️  Missing audio! Run drive_uploader.py on your PC first.')

    # Initialize progress.json if missing
    progress_path = os.path.join(DRIVE_ROOT, 'state/progress.json')
    if not os.path.exists(progress_path):
        progress = {
            'version': 2,
            'total_scenes': len(scenes),
            'engine': 'wan2gp',
            'scenes': {f'scene_{i+1:04d}': 'pending' for i in range(len(scenes))},
            'updated_at': time.time()
        }
        with open(progress_path, 'w') as f:
            json.dump(progress, f, indent=2)
        print(f'✅ Initialized progress.json ({len(scenes)} scenes)')
    else:
        with open(progress_path) as f:
            p = json.load(f)
        done = sum(1 for s in p['scenes'].values() if s == 'done')
        total = p['total_scenes']
        print(f'✅ Progress: {done}/{total} scenes done ({done/total*100:.1f}%)')

## Cell 7 — 🚀 Run the Batch Worker

In [ ]:
# CELL 7: Start the distributed worker loop!
# This runs until all scenes are done or the Colab session expires.
#
# Options:
#   MODEL: 'wan21' (default, fast) or 'ltx2' (with audio support)
#   MAX_SCENES: Set to a number to limit scenes per session, or None for all

import sys
sys.path.insert(0, '/content')

DRIVE_ROOT = '/content/drive/MyDrive/AnimeFactory'
MODEL = 'wan21'      # Change to 'ltx2' for LTX-2 model
MAX_SCENES = None    # Set to e.g. 10 to limit per session

from wan2gp_worker import run_worker
run_worker(DRIVE_ROOT, model=MODEL, max_scenes=MAX_SCENES)

## Cell 8 — 🎞️ Final Stitch (when ALL scenes are done)

In [ ]:
# CELL 8: Stitch all completed scenes into final video
import os, subprocess

DRIVE_ROOT = '/content/drive/MyDrive/AnimeFactory'
scenes_dir = os.path.join(DRIVE_ROOT, 'outputs', 'scenes')
final_path = os.path.join(DRIVE_ROOT, 'outputs', 'final_movie.mp4')

scene_files = sorted([
    os.path.join(scenes_dir, f)
    for f in os.listdir(scenes_dir)
    if f.endswith('.mp4')
])

print(f'Found {len(scene_files)} completed scenes')

# Create concat list
list_file = os.path.join(DRIVE_ROOT, 'outputs', 'concat_list.txt')
with open(list_file, 'w') as f:
    for sf in scene_files:
        f.write(f\"file '{sf}'\\n\")

print('Stitching final movie...')
subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
    '-i', list_file, '-c', 'copy', final_path
], check=True)

size_mb = os.path.getsize(final_path) / 1e6
print(f'✅ Final movie saved: {final_path} ({size_mb:.1f} MB)')